# Architecture flow

Runs `generate_architecture_story` (src/features/architecture/service.py) over a saved repo
context and dumps the resulting architecture graph as JSON into `outputs/architecture`.

**Step 1 (terminal, once):** pull + save the data
```
poetry run python scripts/dump_repo_context.py <repo_id>
```

In [1]:
import glob
import json
from pathlib import Path


# --- Load the most recently saved repo_context_*.json ---
NOTEBOOKS = Path("notebooks") if (Path("notebooks") / "data").exists() else Path(".")
files = sorted(
    glob.glob(str(NOTEBOOKS / "data" / "repo_context_*.json")),
    key=lambda p: Path(p).stat().st_mtime,
    reverse=True,
)
if not files:
    raise FileNotFoundError(
        "No repo_context_*.json found. Run first:\n"
        "  poetry run python scripts/dump_repo_context.py <repo_id>"
    )

# Or pin a specific file: DATA_PATH = NOTEBOOKS / "data" / "repo_context_<id>.json"
DATA_PATH = Path(files[0])
print(f"Loading: {DATA_PATH}")
context = json.loads(DATA_PATH.read_text())

Loading: data/repo_context_743b88e2-f981-4930-95b5-a0c9359f5b2e.json


In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # repo root (notebook runs from notebooks/)
from src.features.architecture.service import generate_architecture_story

ERROR:root:SECRET_KEY is not set. A secure SECRET_KEY is required to sign auth tokens in STAGING/PRODUCTION.
ERROR:root:SECRET_KEY is not set. A secure SECRET_KEY is required to sign auth tokens in STAGING/PRODUCTION.


In [3]:
# --- Prepare inputs for the architecture flow ---
# generate_architecture_story expects file paths and edge dicts keyed by path.
files = [n["file_path"] for n in context["file_nodes"] if n.get("file_path")]
edges = [
    {"source": e["source"], "target": e["target"]}
    for e in context["file_edges"]
]
print(f"Prepared files={len(files)} edges={len(edges)}")

Prepared files=200 edges=131


In [4]:
# Use await (not asyncio.run) because Jupyter already has an event loop.
result = await generate_architecture_story(files=files, edges=edges)
print(json.dumps(result.model_dump(), indent=2)[:2000], "...")
print(f"Summary: {result.summary}")

{
  "summary": "This codebase follows a layered architecture where the UI frontend interacts solely with the API layer to fetch and submit data. The API layer orchestrates business logic and data access, while business logic handles core workflows and may invoke external services. All layers rely on shared utilities and configuration, and tests span across multiple layers to ensure correctness and integration.",
  "root": {
    "id": "root",
    "label": "Architecture",
    "type": "system",
    "description": "This codebase follows a layered architecture where the UI frontend interacts solely with the API layer to fetch and submit data. The API layer orchestrates business logic and data access, while business logic handles core workflows and may invoke external services. All layers rely on shared utilities and configuration, and tests span across multiple layers to ensure correctness and integration.",
    "file_path": null,
    "insight": null,
    "children": [
      {
        "id":

In [5]:
# --- Dump the architecture graph to outputs/architecture ---
OUT_DIR = Path("outputs/architecture")
OUT_DIR.mkdir(parents=True, exist_ok=True)

repo_id = context.get("connected_repo_id", "unknown")
OUT_PATH = OUT_DIR / f"architecture_{repo_id}.json"
OUT_PATH.write_text(json.dumps(result.model_dump(), indent=2))
print(f"Wrote: {OUT_PATH} ({OUT_PATH.stat().st_size} bytes)")

Wrote: outputs/architecture/architecture_743b88e2-f981-4930-95b5-a0c9359f5b2e.json (113600 bytes)
